Rúbrica: 

A continuación se muestra la rúbrica con la que se va a corregir el examen:


| Apartado/Criterio | Ponderación | 
| :-- | --- | 
| Ej. 1.1. Ha tratado de manera adecuada los datos de las columnas. | 1 | 
| Ej. 1.1. Ha seguido un criterio adecuado para elegir las entradas del problema. | 1 |
| Ej. 1.2. Ha diseñado bien la red y el sistema de entrenamiento | 1 | 
| Ej. 1.2. Ha dimensionado bien la red neuronal. | 1 | 
| Ej. 1.2. Ha hecho modificaciones coherentes para conseguir un mejor resultado. | 0,5 | 
| Ej. 1.2. Ha usado su experiencia para valorar si el resultado es válido o no. | 1 | 
| Ej. 2.1. Ha cargado adecuadamente los datos. | 0,5 | 
| Ej. 2.2. Ha diseñado bien la red y el sistema de entrenamiento | 1 | 
| Ej. 2.2. Ha dimensionado bien la red neuronal. | 1 | 
| Ej. 2.2. Ha hecho modificaciones coherentes para conseguir un mejor resultado. | 1 | 
| Ej. 2.2. Ha usado su experiencia para valorar si el resultado es válido o no. | 1 | 



In [1009]:
from os import listdir
from numpy import asarray
from numpy import save
import tensorflow as tf
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from tensorflow import keras
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVR
from sklearn.metrics import classification_report
import datetime
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

## Ejercicio 1

In [1010]:
df_vuelos = pd.read_csv('vuelos_pakistan.csv')

df_vuelos.head()

,Flight_ID,Date,Month,Day_of_Week,Departure_City,Arrival_City,Route_Type,Aircraft_Type,Flight_Duration_Minutes,Passengers,...,Load_Factor_%,Ticket_Price_USD,Delay_Minutes,Delay_Category,On_Time_Status,Weather_Condition,Fuel_Consumption,CO2_Emissions,Customer_Rating,Customer_Feedback
0,PK2026_0001,2026-06-09,June,Tuesday,Jeddah,Islamabad,International,Airbus A320,83.0,120,...,66.67,1140.0,220,Severe,Delayed,Clear,6265l,15662.5kg,4.1,Dreadful customer support
1,PK2026_0002,2026-08-12,August,Wednesday,Dubai,Kuala Lumpur,International,Airbus A320,284.0,179,...,99.44,773.0,27,Minor,Delayed,NaN,3516l,8790.0kg,3.6,"Standard flight, nothing special"
2,PK2026_0003,2026-04-20,April,Monday,Doha,Lahore,International,ATR 72,333.0,69,...,98.57,155.0,176,Severe,Delayed,Fog,13538l,33845.0kg,3.0,Tardy arrival but very cozy
3,PK2026_0004,2026-12-07,December,Monday,Jeddah,Lahore,International,Boeing 777,330.0,291,...,83.14,1237.0,87,Moderate,Delayed,NaN,18850l,47125.0kg,NaN,Mediocre experience overall
4,PK2026_0005,2026-05-04,May,Monday,Lahore,Doha,International,Boeing 737,283.0,159,...,99.38,141.0,82,Moderate,Delayed,NaN,13474l,33685.0kg,3.0,Behind schedule but quite relaxed


In [1011]:
df_vuelos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Flight_ID                800 non-null    object 
 1   Date                     800 non-null    object 
 2   Month                    800 non-null    object 
 3   Day_of_Week              800 non-null    object 
 4   Departure_City           800 non-null    object 
 5   Arrival_City             800 non-null    object 
 6   Route_Type               800 non-null    object 
 7   Aircraft_Type            800 non-null    object 
 8   Flight_Duration_Minutes  725 non-null    float64
 9   Passengers               800 non-null    int64  
 10  Seat_Capacity            800 non-null    int64  
 11  Load_Factor_%            800 non-null    float64
 12  Ticket_Price_USD         786 non-null    float64
 13  Delay_Minutes            800 non-null    int64  
 14  Delay_Category           8

In [1012]:
df_vuelos['Route_Type'].unique()

array(['International', 'Domestic'], dtype=object)

In [1013]:
day_list = []
month_list = []
year_list = []
c02_list = []
fuel_list = []

for index, row in df_vuelos.iterrows():
    divided_date = row['Date'].split('-')
    year_list.append(int(divided_date[0]))
    month_list.append(int(divided_date[1]))
    day_list.append(int(divided_date[2]))
    c02_list.append(float(row['CO2_Emissions'].replace('kg', '')))
    fuel_list.append(float(row['Fuel_Consumption'].replace('l', '')))

df_vuelos['Year'] = year_list
df_vuelos['Month'] = month_list
df_vuelos['Day'] = day_list
df_vuelos['CO2_Emissions'] = c02_list
df_vuelos['Fuel_Consumption'] = fuel_list

In [1014]:
df_vuelos['Day_of_Week'].replace({'Tuesday' : 2, 'Wednesday': 3, 'Monday' : 1, 'Saturday' : 7, 'Friday' : 5, 'Sunday' : 6,
       'Thursday' : 4}, inplace=True)

df_vuelos.head()

/tmp/ipykernel_49164/2250224499.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_vuelos['Day_of_Week'].replace({'Tuesday' : 2, 'Wednesday': 3, 'Monday' : 1, 'Saturday' : 7, 'Friday' : 5, 'Sunday' : 6,
/tmp/ipykernel_49164/2250224499.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_vuelos

,Flight_ID,Date,Month,Day_of_Week,Departure_City,Arrival_City,Route_Type,Aircraft_Type,Flight_Duration_Minutes,Passengers,...,Delay_Minutes,Delay_Category,On_Time_Status,Weather_Condition,Fuel_Consumption,CO2_Emissions,Customer_Rating,Customer_Feedback,Year,Day
0,PK2026_0001,2026-06-09,6,2,Jeddah,Islamabad,International,Airbus A320,83.0,120,...,220,Severe,Delayed,Clear,6265.0,15662.5,4.1,Dreadful customer support,2026,9
1,PK2026_0002,2026-08-12,8,3,Dubai,Kuala Lumpur,International,Airbus A320,284.0,179,...,27,Minor,Delayed,NaN,3516.0,8790.0,3.6,"Standard flight, nothing special",2026,12
2,PK2026_0003,2026-04-20,4,1,Doha,Lahore,International,ATR 72,333.0,69,...,176,Severe,Delayed,Fog,13538.0,33845.0,3.0,Tardy arrival but very cozy,2026,20
3,PK2026_0004,2026-12-07,12,1,Jeddah,Lahore,International,Boeing 777,330.0,291,...,87,Moderate,Delayed,NaN,18850.0,47125.0,NaN,Mediocre experience overall,2026,7
4,PK2026_0005,2026-05-04,5,1,Lahore,Doha,International,Boeing 737,283.0,159,...,82,Moderate,Delayed,NaN,13474.0,33685.0,3.0,Behind schedule but quite relaxed,2026,4


In [1015]:
df_vuelos.corr(numeric_only=True)['Ticket_Price_USD'].abs().sort_values(ascending=False)[1:]
# df_vuelos.head()

Day                        0.056945
Month                      0.056821
Delay_Minutes              0.032351
Passengers                 0.026436
Customer_Rating            0.025094
Fuel_Consumption           0.018277
CO2_Emissions              0.018277
Seat_Capacity              0.012411
Day_of_Week                0.007677
Load_Factor_%              0.004942
Flight_Duration_Minutes    0.001597
Year                            NaN
Name: Ticket_Price_USD, dtype: float64

In [1016]:
df_vuelos.drop(columns=['Flight_ID', 'Date', 'Weather_Condition', 'Year', 'Customer_Feedback'], inplace=True)

In [1017]:
df_vuelos.dropna(inplace=True)

In [1018]:
df_vuelos = pd.get_dummies(df_vuelos,dtype=int)

In [1019]:
df_vuelos.head()

,Month,Day_of_Week,Flight_Duration_Minutes,Passengers,Seat_Capacity,Load_Factor_%,Ticket_Price_USD,Delay_Minutes,Fuel_Consumption,CO2_Emissions,...,Aircraft_Type_ATR 72,Aircraft_Type_Airbus A320,Aircraft_Type_Boeing 737,Aircraft_Type_Boeing 777,Delay_Category_Minor,Delay_Category_Moderate,Delay_Category_No Delay,Delay_Category_Severe,On_Time_Status_Delayed,On_Time_Status_On Time
0,6,2,83.0,120,180,66.67,1140.0,220,6265.0,15662.5,...,0,1,0,0,0,0,0,1,1,0
1,8,3,284.0,179,180,99.44,773.0,27,3516.0,8790.0,...,0,1,0,0,1,0,0,0,1,0
2,4,1,333.0,69,70,98.57,155.0,176,13538.0,33845.0,...,1,0,0,0,0,0,0,1,1,0
4,5,1,283.0,159,160,99.38,141.0,82,13474.0,33685.0,...,0,0,1,0,0,1,0,0,1,0
5,1,1,433.0,172,180,95.56,911.0,192,5207.0,13017.5,...,0,1,0,0,0,0,0,1,1,0


In [1020]:
df_vuelos.shape

(599, 40)

In [1021]:
X = df_vuelos.drop(columns=['Ticket_Price_USD'])
y = df_vuelos['Ticket_Price_USD'].to_frame()

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [1022]:
X.shape

(599, 39)

In [1023]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [1024]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(25, activation='relu'))
model.add(keras.layers.Dense(16, activation='relu'))
model.add(keras.layers.Dense(1))

In [1025]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(22, activation='relu'))
model.add(keras.layers.Dense(14, activation='relu'))
model.add(keras.layers.Dense(1))

In [1026]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(26, activation='relu'))
model.add(keras.layers.Dense(17, activation='relu'))
model.add(keras.layers.Dense(1))

In [1027]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(32, activation='relu'))
model.add(keras.layers.Dense(16, activation='relu'))
model.add(keras.layers.Dense(1))

In [1028]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(26, activation='relu'))
model.add(keras.layers.Dense(16, activation='relu'))
model.add(keras.layers.Dense(1))

In [1029]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(26, activation='relu'))
model.add(keras.layers.Dense(12, activation='relu'))
model.add(keras.layers.Dense(1))

In [1030]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(26, activation='relu'))
model.add(keras.layers.Dense(10, activation='relu'))
model.add(keras.layers.Dense(1))

In [1031]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(22, activation='relu'))
model.add(keras.layers.Dense(10, activation='relu'))
model.add(keras.layers.Dense(1))

In [1032]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(16, activation='relu'))
model.add(keras.layers.Dense(10, activation='relu'))
model.add(keras.layers.Dense(1))

In [1033]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(20, activation='relu'))
model.add(keras.layers.Dense(12, activation='relu'))
model.add(keras.layers.Dense(1))

In [1034]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(20, activation='relu'))
model.add(keras.layers.Dense(13, activation='relu'))
model.add(keras.layers.Dense(1))

In [1035]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(26, activation='relu'))
model.add(keras.layers.Dense(14, activation='relu'))
model.add(keras.layers.Dense(1))

In [1036]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(27, activation='relu'))
model.add(keras.layers.Dense(15, activation='relu'))
model.add(keras.layers.Dense(1))

In [1037]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(22, activation='relu'))
model.add(keras.layers.Dense(14, activation='relu'))
model.add(keras.layers.Dense(1))

In [1038]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(27, activation='relu'))
model.add(keras.layers.Dense(17, activation='relu'))
model.add(keras.layers.Dense(1))

In [1039]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(26, activation='relu'))
model.add(keras.layers.Dense(12, activation='relu'))
model.add(keras.layers.Dense(1))

In [1040]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(26, activation='relu'))
model.add(keras.layers.Dense(17, activation='relu'))
model.add(keras.layers.Dense(1))

In [1041]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(26, activation='relu'))
model.add(keras.layers.Dense(16, activation='relu'))
model.add(keras.layers.Dense(1))

In [1042]:
model.summary()

Model: "sequential_250"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_749 (Dense)               │ (None, 26)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_750 (Dense)               │ (None, 16)             │           432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_751 (Dense)               │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,489 (5.82 KB)

 Trainable params: 1,489 (5.82 KB)

 Non-trainable params: 0 (0.00 B)

In [1043]:
model.compile(loss='mean_absolute_error', metrics=['mae'], optimizer = keras.optimizers.SGD(learning_rate=0.01))

In [1044]:
# model.compile(loss='mean_absolute_error', metrics=['mse'], optimizer = keras.optimizers.SGD(learning_rate=0.01))

In [1045]:
history = model.fit(X_train, y_train, epochs=40 ,validation_split=0.1)

Epoch 1/40
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 793.2856 - mae: 793.2856 - val_loss: 815.9446 - val_mae: 815.9446
Epoch 2/40
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 791.1182 - mae: 791.1182 - val_loss: 813.0295 - val_mae: 813.0295
Epoch 3/40
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 786.6691 - mae: 786.6691 - val_loss: 806.4727 - val_mae: 806.4727
Epoch 4/40
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 775.0876 - mae: 775.0876 - val_loss: 787.1685 - val_mae: 787.1685
Epoch 5/40
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 731.5952 - mae: 731.5952 - val_loss: 695.3918 - val_mae: 695.3918
Epoch 6/40
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 539.2964 - mae: 539.2964 - val_loss: 470.8792 - val_mae: 470.8792
Epoch 7/40
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 370.5484 - mae: 370.5484 - val_loss: 377.1274 - val_mae: 377.1274
Epoch 8/40
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 345.0690 - mae: 345.0690 - val_loss: 399.5773 - val_mae: 399.5773
Epoch 9/

In [ ]:
#Me estoy volviendo loco, he logrado que de -0.10, pero no se que le pasa que sin tocar de repente ha subido una barbaridad
y_pred = model.predict(X_test)
r2_score(y_test,y_pred)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 


-0.2674063444137573

## Ejercicio 2

In [1164]:
#Esta da peor pero la dejo porque la idea es tremenda 
folders = listdir('cartas')
photos = []
labels = []
photoLimit = 60

for idx, folder in enumerate(folders):
    counter = 0
    
    for file in listdir('cartas/' + folder ):
        photo = load_img('cartas/'+folder+'/'+file, target_size=(40,40), color_mode='grayscale')
        photo = img_to_array(photo)
        if(counter < photoLimit):
            photos.append(photo)
            labels.append(float(idx))
            counter = counter + 1    

In [1211]:
folders = listdir('cartas')
photos = []
labels = []

for idx, folder in enumerate(folders):
    counter = 0
    
    for file in listdir('cartas/' + folder ):
        photo = load_img('cartas/'+folder+'/'+file, target_size=(40,40), color_mode='grayscale')
        photo = img_to_array(photo)
        photos.append(photo)
        labels.append(float(idx))
  

In [1212]:
print(len(folders))

53


In [1213]:
photos = asarray(photos).astype('float32') / 255
photos = photos.reshape(len(photos), -1)
labels = asarray(labels)

In [1214]:
X_train, X_test, y_train, y_test = train_test_split(photos, labels, test_size=0.2,)

In [1215]:
X_train.shape[1:]

(1600,)

In [1216]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(10000, activation='relu'))
model.add(keras.layers.Dense(2000, activation='relu'))
model.add(keras.layers.Dense(53, activation='softmax'))

In [1217]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(2000, activation='relu'))
model.add(keras.layers.Dense(600, activation='relu'))
model.add(keras.layers.Dense(53, activation='softmax'))

In [1218]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(1024, activation='relu'))
model.add(keras.layers.Dense(512, activation='relu'))
model.add(keras.layers.Dense(53, activation='softmax'))

In [1219]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(1024, activation='relu'))
model.add(keras.layers.Dense(680, activation='relu'))
model.add(keras.layers.Dense(53, activation='softmax'))

In [1220]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(1024, activation='relu'))
model.add(keras.layers.Dense(676, activation='relu'))
model.add(keras.layers.Dense(53, activation='softmax'))

In [1221]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(1024, activation='relu'))
model.add(keras.layers.Dense(678, activation='relu'))
model.add(keras.layers.Dense(53, activation='softmax'))

In [1222]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(1050, activation='relu'))
model.add(keras.layers.Dense(688, activation='relu'))
model.add(keras.layers.Dense(53, activation='softmax'))

In [1223]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(1000, activation='relu'))
model.add(keras.layers.Dense(650, activation='relu'))
model.add(keras.layers.Dense(53, activation='softmax'))

In [1224]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(1000, activation='relu'))
model.add(keras.layers.Dense(600, activation='relu'))
model.add(keras.layers.Dense(53, activation='softmax'))

In [1225]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(1000, activation='relu'))
model.add(keras.layers.Dense(666, activation='relu'))
model.add(keras.layers.Dense(53, activation='softmax'))

In [1226]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(1000, activation='relu'))
model.add(keras.layers.Dense(640, activation='relu'))
model.add(keras.layers.Dense(53, activation='softmax'))

In [1227]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(1000, activation='relu'))
model.add(keras.layers.Dense(655, activation='relu'))
model.add(keras.layers.Dense(53, activation='softmax'))

In [1228]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(1000, activation='relu'))
model.add(keras.layers.Dense(645, activation='relu'))
model.add(keras.layers.Dense(53, activation='softmax'))

In [1229]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(1000, activation='relu'))
model.add(keras.layers.Dense(650, activation='relu'))
model.add(keras.layers.Dense(53, activation='softmax'))

In [1230]:
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(1000, activation='relu'))
model.add(keras.layers.Dense(645, activation='relu'))
model.add(keras.layers.Dense(53, activation='softmax'))

In [1231]:
model.summary()

Model: "sequential_356"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_1067 (Dense)              │ (None, 1000)           │     1,601,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1068 (Dense)              │ (None, 645)            │       645,645 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1069 (Dense)              │ (None, 53)             │        34,238 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,280,883 (8.70 MB)

 Trainable params: 2,280,883 (8.70 MB)

 Non-trainable params: 0 (0.00 B)

In [1232]:
model.compile(loss='sparse_categorical_crossentropy', metrics=['accuracy'], optimizer = keras.optimizers.SGD(learning_rate=0.01) )

In [1233]:
history = model.fit(X_train,y_train, epochs=18, validation_split=0.1)

Epoch 1/18
172/172 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.0594 - loss: 3.8585 - val_accuracy: 0.1131 - val_loss: 3.6944
Epoch 2/18
172/172 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.1281 - loss: 3.5535 - val_accuracy: 0.1246 - val_loss: 3.4137
Epoch 3/18
172/172 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.1581 - loss: 3.3120 - val_accuracy: 0.1541 - val_loss: 3.1991
Epoch 4/18
172/172 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.1838 - loss: 3.1336 - val_accuracy: 0.1754 - val_loss: 3.1054
Epoch 5/18
172/172 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.2141 - loss: 3.0044 - val_accuracy: 0.1951 - val_loss: 2.9998
Epoch 6/18
172/172 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.2427 - loss: 2.9016 - val_accuracy: 0.2246 - val_loss: 2.8649
Epoch 7/18
172/172 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.2682 - loss: 2.8117 - val_accuracy: 0.2115 - val_loss: 2.8480
Epoch 8/18
172/172 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.2902 - loss: 2.7332 - val_accuracy: 0.

In [1234]:
#Hay que tomar en cuenta que tiene 53 clases, un ramdon no subiria de 0.02
model.evaluate(X_test, y_test)

48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3095 - loss: 2.6382


[2.6382343769073486, 0.30950820446014404]